# Column Timing Audit — User Workshop

**This is your audit.** Work from the top, one cell at a time. Read each question, run the evidence cell, then make and record your own bin choice in the next code cell. The notebook does not pre-fill the classifications.

The earlier `docs/COLUMNS.md` contains AI-generated preliminary suggestions. Treat them only as material to review; they are not your decisions. Do not copy a bin without understanding the evidence and reason.

By the end, you should be able to explain when each field is known, whether it can change, and whether it is safe for the intended decision time. The public export alone may not answer that: the source article says data values were extracted from booking/change records as of the day before arrival.
**Full paper reference:** See [COLUMN_SOURCE_REFERENCE.md](../docs/COLUMN_SOURCE_REFERENCE.md) for the full definitions and profiles of all 32 fields.


## How to work through the notebook

1. Run each code cell in order, using **Shift+Enter**.
2. Before each decision cell, read the evidence output and the source note in `data/README.md` or the linked paper.
3. In the decision cell, replace `None` with your bin (`1`, `2`, or `3`) and write your reason in your own words.
4. If the source does not settle timing, choose the cautious bin and write what must be verified.
5. The final cell checks completeness and writes `docs/COLUMNS_USER_REVIEW.md` from your recorded choices. It will not write a report while any field is unanswered.

**Bins:** 1 = known at booking and does not change; 2 = amendable or timing uncertain; 3 = after the fact / outcome-related. These definitions follow the project instructions. A field needed to group a hotel-night can still be an operational key; say so in your reason.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

### Cell 1 — Find and load the project data

Run this cell. It locates the repository from the notebook's current folder and reads the original CSV without changing values or rows. If the file is missing, stop and tell me what the error says before downloading or editing anything.


In [2]:
project_root = Path.cwd()
data_path = project_root / "data" / "hotels.csv"

# If Jupyter opened in notebooks/, look one folder up.
if not data_path.exists():
    project_root = project_root.parent
    data_path = project_root / "data" / "hotels.csv"

if not data_path.exists():
    raise FileNotFoundError("Could not find data/hotels.csv. Open Jupyter from the project folder.")

bookings = pd.read_csv(data_path)
print("Loaded rows:", len(bookings))
print("Loaded columns:", len(bookings.columns))
print("Data file:", data_path)


Loaded rows: 119390
Loaded columns: 32
Data file: c:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06\data\hotels.csv


### Cell 2 — Confirm the data shape and field names

Check that the downloaded data matches the brief. This cell reports metadata only; it does not classify fields for you.


In [3]:
print("Rows:", len(bookings))
print("Columns:", len(bookings.columns))

column_names = list(bookings.columns)
column_types = list(bookings.dtypes)
column_info = pd.DataFrame({
    "column_name": column_names,
    "data_type": column_types,
})
display(column_info)


Rows: 119390
Columns: 32


,column_name,data_type
0,hotel,object
1,is_canceled,int64
2,lead_time,int64
3,arrival_date_year,int64
4,arrival_date_month,object
5,arrival_date_week_number,int64
6,arrival_date_day_of_month,int64
7,stays_in_weekend_nights,int64
8,stays_in_week_nights,int64
9,adults,int64


### Cell 3 — Build the arrival-date split

The scoring periods are based on arrival date. This cell shows row counts and date limits only. It deliberately does not inspect cancellation outcomes in the proof period.


In [4]:
arrival_date_text = (
    bookings["arrival_date_year"].astype(str)
    + "-"
    + bookings["arrival_date_month"]
    + "-"
    + bookings["arrival_date_day_of_month"].astype(str)
)
arrival_date = pd.to_datetime(arrival_date_text, format="%Y-%B-%d")

train_mask = arrival_date < "2016-07-01"
choose_mask = (arrival_date >= "2016-07-01") & (arrival_date < "2017-01-01")
proof_mask = (arrival_date >= "2017-01-01") & (arrival_date < "2017-09-01")

period_rows = []
period_rows.append({"period": "train", "rows": train_mask.sum()})
period_rows.append({"period": "choose", "rows": choose_mask.sum()})
period_rows.append({"period": "proof", "rows": proof_mask.sum()})
period_summary = pd.DataFrame(period_rows)
display(period_summary)

# Only train and choose rows are used for the examples in this notebook.
train_choose_mask = train_mask | choose_mask
train_choose = bookings.loc[train_choose_mask].copy()
print("Rows used for examples:", len(train_choose))


,period,rows
0,train,49157
1,choose,29546
2,proof,40687


Rows used for examples: 78703


### Cell 4 — Keep the two hotels distinct

Look at the hotel labels and counts. Decide in your own words whether `hotel` is a feature, a grouping key, or both for this project. Do not infer different operating policies just from different cancellation rates.


In [5]:
hotel_counts = train_choose["hotel"].value_counts()
display(hotel_counts)


hotel
City Hotel      51822
Resort Hotel    26881
Name: count, dtype: int64

### Cell 5 — Prepare your decision notebook

The dictionary starts empty. Each group below adds your answers. Do not fill in a bin until you have reviewed that group's evidence and written a reason.


In [6]:
my_decisions = {}


### Cell group 6 — Group 1 — Property and outcome

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which field identifies the property? Which field is the target/outcome, so it must never be a predictor? Explain the difference.

### Paper evidence for these fields

### 1. `hotel` — Hotel property (CSV category)

- **Paper context:** H1 is the resort hotel in Algarve; H2 is the city hotel in Lisbon. The article reports 40,060 H1 bookings and 79,330 H2 bookings, 119,390 in total. Property/customer identifiers were removed or anonymized.
- **Source / construction:** This is the property identifier that separates the two supplied datasets; it is not one of the 31 Table 1 booking variables.
- **Published profile:** H1 40,060 rows; H2 79,330 rows.
- **Questions for your decision:** Is `hotel` an input, an essential grouping key for one decision per hotel-night, or both? Does a property identifier change for an existing booking? Does a difference in hotel cancellation rates prove a different booking rule? What evidence would establish per-hotel capacity or policy differences?

### 2. `is_canceled` — IsCanceled (categorical / binary)

- **Paper definition:** Indicates whether a booking was canceled: 1 or 0.
- **Source / engineering:** BO.
- **Published profile:** H1: 0 = 28,938; 1 = 11,122. H2: 0 = 46,228; 1 = 33,102.
- **Questions for your decision:** This is the prediction target or label, not a usable input. The paper describes a cancellation indicator; the project scoring contract says the target's 1 includes no-shows. How will you explain the contract's interpretation alongside the paper's separate `No-Show` status? How will you keep proof-period labels out of the early audit and tuning?


In [7]:
show_columns = ['hotel', 'is_canceled']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,hotel,object,0,"Resort Hotel, City Hotel"
1,is_canceled,int64,0,"0, 1"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "hotel": {"bin": None, "reason": ""},
    "is_canceled": {"bin": None, "reason": ""},
})


### Cell group 7 — Group 2 — Booking lead time and arrival calendar

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which values are known when the booking is first made? Can the arrival date change later? Which values are keys needed to group the action by hotel-night?

### Paper evidence for these fields

### 3. `lead_time` — LeadTime (integer)

- **Paper definition:** Number of days from the date the booking was entered in the PMS to the arrival date.
- **Source / engineering:** BO and BL; calculated by subtracting the booking-entry date from the arrival date.
- **Published profile:** H1: 92.68, 97.29, 0, 10, 57, 155, 737. H2: 109.74, 110.95, 0, 23, 74, 163, 629.
- **Questions for your decision:** Is the booking-entry date known when the reservation is created? Can an arrival-date amendment change the final lead time? Does the CSV retain the original booking-entry value or only the day-before-arrival version? What point-in-time value would the rule use?

### 4. `arrival_date_year` — ArrivalDateYear (integer)

- **Paper definition:** Year of the arrival date.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 2016.12, 0.72, 2015, 2016, 2016, 2017, 2017. H2: 2016.17, 0.70, 2015, 2016, 2016, 2017, 2017.
- **Questions for your decision:** Is this a predictor or part of the hotel-night key? Could it change if the arrival date changes? If you need it for grouping, which dated snapshot provides the operative arrival date?

### 5. `arrival_date_month` — ArrivalDateMonth (categorical)

- **Paper definition:** Month of arrival, with January through December as the 12 categories.
- **Source / engineering:** BO and BL.
- **Published profile:** 12 unique values at each hotel. H1 top counts: August 4,894; July 4,573; April 3,609; May 3,559. H2: August 8,983; May 8,232; July 8,088; June 7,894.
- **Questions for your decision:** Does the month stay fixed if arrival is amended? Is it a seasonal input or only a grouping field? How should it be derived if the hotel-night date is updated?

### 6. `arrival_date_week_number` — ArrivalDateWeekNumber (integer)

- **Paper definition:** Week number of the arrival date.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 27.14, 14.01, 1, 16, 28, 38, 53. H2: 27.18, 13.40, 1, 17, 27, 38, 53.
- **Questions for your decision:** Is this only a date-derived field? Could it change with an arrival-date amendment? Do you need this separate calendar representation if you already use the full arrival date?

### 7. `arrival_date_day_of_month` — ArrivalDateDayOfMonth (integer)

- **Paper definition:** Day number within the arrival month.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 15.82, 8.88, 1, 8, 16, 24, 31. H2: 15.79, 8.73, 1, 8, 16, 23, 31.
- **Questions for your decision:** Is this a predictor or part of the decision key? Can it change if the reservation's arrival is amended? What representation will reliably identify one arrival night?


In [8]:
show_columns = ['lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,lead_time,int64,0,"342, 737, 7, 13"
1,arrival_date_year,int64,0,"2015, 2016"
2,arrival_date_month,object,0,"July, August, September, October"
3,arrival_date_week_number,int64,0,"27, 28, 29, 30"
4,arrival_date_day_of_month,int64,0,"1, 2, 3, 4"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "lead_time": {"bin": None, "reason": ""},
    "arrival_date_year": {"bin": None, "reason": ""},
    "arrival_date_month": {"bin": None, "reason": ""},
    "arrival_date_week_number": {"bin": None, "reason": ""},
    "arrival_date_day_of_month": {"bin": None, "reason": ""},
})


### Cell group 8 — Group 3 — Length of stay

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Could a guest amend the stay length? Does the final export prove the original booking-time value?

### Paper evidence for these fields

### 8. `stays_in_weekend_nights` — StaysInWeekendNights (integer)

- **Paper definition:** Number of Saturday/Sunday nights the guest stayed or booked to stay.
- **Source / engineering:** BO and BL; calculated by counting weekend nights among total nights.
- **Published profile:** H1: 1.19, 1.15, 0, 0, 1, 2, 19. H2: 0.80, 0.89, 0, 0, 1, 2, 16.
- **Questions for your decision:** Is this the original requested stay or a later amended stay? The paper says stay duration can change. If the cost contract uses the booked stay value, which timestamp's stay length should be used?

### 9. `stays_in_week_nights` — StaysInWeekNights (integer)

- **Paper definition:** Number of Monday–Friday nights the guest stayed or booked to stay.
- **Source / engineering:** BO and BL; calculated from total nights.
- **Published profile:** H1: 3.13, 2.46, 0, 1, 3, 5, 50. H2: 2.18, 1.46, 0, 1, 2, 3, 41.
- **Questions for your decision:** Could this be amended before arrival? How would a changed departure date affect the final value? Which version should be used for the project cost formula and why?


In [9]:
show_columns = ['stays_in_weekend_nights', 'stays_in_week_nights']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,stays_in_weekend_nights,int64,0,"0, 1, 2, 4"
1,stays_in_week_nights,int64,0,"0, 1, 2, 3"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "stays_in_weekend_nights": {"bin": None, "reason": ""},
    "stays_in_week_nights": {"bin": None, "reason": ""},
})


### Cell group 9 — Group 4 — Party size

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Could the party composition change after booking? What does the source say about changes to guest attributes?

### Paper evidence for these fields

### 10. `adults` — Adults (integer)

- **Paper definition:** Number of adults.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 1.87, 0.70, 0, 2, 2, 2, 55. H2: 1.85, 0.51, 0, 2, 2, 2, 4.
- **Questions for your decision:** Can the guest party size change after the reservation is created? Does the article's caution about guest attributes changing apply? At what snapshot time is this count available?

### 11. `children` — Children (integer)

- **Paper definition:** Number of children.
- **Source / engineering:** BO and BL; sum of both payable and non-payable children.
- **Published profile:** H1: 0.13, 0.45, 0, 0, 0, 0, 10. H2: 0.091, 0.37, 0, 0, 0, 0, 3.
- **Questions for your decision:** Can this count change? Does its combined payable/non-payable definition affect how you interpret zeros and extreme values? Is it known at initial booking or only in a later snapshot?

### 12. `babies` — Babies (integer)

- **Paper definition:** Number of babies.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 0.014, 0.12, 0, 0, 0, 0, 2. H2: 0.0049, 0.084, 0, 0, 0, 0, 10.
- **Questions for your decision:** Can the party composition change after booking? What would you check about the unusually high H2 maximum of 10 before using or correcting it?


In [10]:
show_columns = ['adults', 'children', 'babies']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,adults,int64,0,"2, 1, 3, 4"
1,children,float64,4,"0.0, 1.0, 2.0, 10.0"
2,babies,int64,0,"0, 1, 2, 10"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "adults": {"bin": None, "reason": ""},
    "children": {"bin": None, "reason": ""},
    "babies": {"bin": None, "reason": ""},
})


### Cell group 10 — Group 5 — Meal, geography, and sales categories

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** When would the hotel know each value? Does the source say country is always known before check-in? Are the category values proven immutable at booking time?

### Paper evidence for these fields

### 13. `meal` — Meal (categorical)

- **Paper definition:** Meal package. Undefined/SC means no meal package; BB means bed and breakfast; HB means half-board (breakfast plus one other meal, usually dinner); FB means full-board (breakfast, lunch, and dinner).
- **Source / engineering:** BO, BL, and ML (Meals table).
- **Published profile:** H1 has 5 unique values; top counts: BB 30,005; HB 8,046; Undefined 1,169; FB 754. H2 has 4; top counts: BB 62,305; SC 10,564; HB 6,417; FB 44.
- **Questions for your decision:** When is the meal package selected? Can it be amended? Does BO/BL establish the initial choice or the later value? What source timestamp would answer this?

### 14. `country` — Country (categorical)

- **Paper definition:** Country of origin, represented using ISO 3155-3:2013 codes.
- **Source / engineering:** BO, BL, and NT (Nationalities table).
- **Published profile:** H1 has 125 unique values; top: PRT 17,630; GBR 6,814; ESP 3,957; IRL 2,166. H2 has 166; top: PRT 30,960; FRA 8,804; DEU 6,084; GBR 5,315.
- **Paper timing caution:** The authors say hotels may not know the correct nationality until check-in.
- **Questions for your decision:** When does the hotel actually learn the correct country? Is it recorded from the guest's declared origin, a profile, or a later check-in update? Could it expose post-booking information?

### 15. `market_segment` — MarketSegment (categorical)

- **Paper definition:** Market-segment designation. TA means travel agents; TO means tour operators.
- **Source / engineering:** BO, BL, and MS (Market-segments table).
- **Published profile:** H1 has 6 unique values; top: Online 17,729; Offline 7,472; Direct 6,513; Groups 5,836. H2 has 8; top: Online 38,748; Offline 16,747; Groups 13,975; Direct 6,093.
- **Questions for your decision:** Is the category assigned at booking creation, derived later, or amendable? Does the market-segment table preserve time history? Which categories are in the CSV, and what do they mean in the client's setting?

### 16. `distribution_channel` — DistributionChannel (categorical)

- **Paper definition:** Booking distribution channel. TA means travel agents; TO means tour operators.
- **Source / engineering:** BO, BL, and DC (Distribution-channels table).
- **Published profile:** H1 has 4 unique values; top: TA/TO 28,295; Direct 7,865; Corporate 3,269; Undefined 1. H2 has 5; top: TA/TO 68,945; Direct 6,780; Corporate 3,408; GDS 193.
- **Questions for your decision:** At what moment does the channel become known? Can a reservation transferred or amended by a partner change this field? Does the final export record the original or latest channel?


In [11]:
show_columns = ['meal', 'country', 'market_segment', 'distribution_channel']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,meal,object,0,"BB, FB, HB, SC"
1,country,object,405,"PRT, GBR, USA, ESP"
2,market_segment,object,0,"Direct, Corporate, Online TA, Offline TA/TO"
3,distribution_channel,object,0,"Direct, Corporate, TA/TO, Undefined"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "meal": {"bin": None, "reason": ""},
    "country": {"bin": None, "reason": ""},
    "market_segment": {"bin": None, "reason": ""},
    "distribution_channel": {"bin": None, "reason": ""},
})


### Cell group 11 — Group 6 — Guest history

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Are these based on history strictly before the current booking? What source detail supports your conclusion?

### Paper evidence for these fields

### 17. `is_repeated_guest` — IsRepeatedGuest (categorical / binary)

- **Paper definition:** 1 if the booking name is from a repeat guest, 0 otherwise.
- **Source / engineering:** BO, BL, and C (Customer profile). A profile must be associated with the booking, and its creation date must be before the booking creation date to count as repeated.
- **Published profile:** H1: 0 = 38,282; 1 = 1,778. H2: 0 = 77,298; 1 = 2,032.
- **Questions for your decision:** Does the definition use only history available before the current booking? What happens when no profile is associated? Does the day-before-arrival snapshot preserve that historical test?

### 18. `previous_cancellations` — PreviousCancellations (integer)

- **Paper definition:** Number of the customer's prior bookings canceled before the current booking.
- **Source / engineering:** BO and BL. If there is no associated customer profile, the value is set to 0; otherwise count that profile's earlier canceled bookings.
- **Published profile:** H1: 0.10, 1.34, 0, 0, 0, 0, 26. H2: 0.08, 0.42, 0, 0, 0, 0, 32.
- **Questions for your decision:** Are all counted cancellations strictly before the current booking? Does 0 mean no earlier cancellations, or can it also mean no associated profile? Is that distinction important for your use?

### 19. `previous_bookings_not_canceled` — PreviousBookingsNotCanceled (integer)

- **Paper definition:** Number of the customer's prior bookings that were not canceled before the current booking.
- **Source / engineering:** BO and BL. If no customer profile is associated, value is 0; otherwise count earlier non-canceled bookings for that profile.
- **Published profile:** H1: 0.15, 1.00, 0, 0, 0, 0, 30. H2: 0.13, 1.69, 0, 0, 0, 0, 72.
- **Questions for your decision:** Is the history limited to bookings before this booking? How is a missing profile distinguished from a true zero-history guest? Do you have evidence the history count is available at the decision time?


In [12]:
show_columns = ['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,is_repeated_guest,int64,0,"0, 1"
1,previous_cancellations,int64,0,"0, 1, 2, 3"
2,previous_bookings_not_canceled,int64,0,"0, 1, 2, 3"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "is_repeated_guest": {"bin": None, "reason": ""},
    "previous_cancellations": {"bin": None, "reason": ""},
    "previous_bookings_not_canceled": {"bin": None, "reason": ""},
})


### Cell group 12 — Group 7 — Room types and changes

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which room field is requested and which is assigned? Can the request or assignment change? Does the change count include events after booking creation?

### Paper evidence for these fields

### 20. `reserved_room_type` — ReservedRoomType (categorical)

- **Paper definition:** An anonymized code for the room type reserved.
- **Source / engineering:** BO and BL. The paper lists reserved room type among attributes that can change in the booking change log.
- **Published profile:** H1 has 10 unique values; top: A 23,399; D 7,433; E 4,892; G 1,610. H2 has 8; top: A 62,595; D 11,768; F 1,791; E 1,553.
- **Questions for your decision:** Does the value represent the original request or the latest requested room? Can guests change it? Would the room category be known at the time your decision is made?

### 21. `assigned_room_type` — AssignedRoomType (categorical)

- **Paper definition:** An anonymized code for the room type assigned to the booking. It can differ from the reserved type due to hotel operations (for example, overbooking) or customer request.
- **Source / engineering:** BO and BL.
- **Published profile:** H1 has 11 unique values; top: A 17,046; D 10,339; E 5,638; C 2,214. H2 has 9; top: A 57,007; D 14,983; E 2,168; F 2,018.
- **Questions for your decision:** When is the assignment first made and can it be changed? Might it encode an operational decision made after the oversell recommendation? Could it reveal the result of overbooking itself?

### 22. `booking_changes` — BookingChanges (integer)

- **Paper definition:** Count of changes/amendments from when a booking entered the PMS until check-in or cancellation.
- **Source / engineering:** BO and BL. The count adds unique iterations that change persons, arrival date, nights, reserved room type, or meal.
- **Published profile:** H1: 0.29, 0.73, 0, 0, 0, 0, 17. H2: 0.19, 0.61, 0, 0, 0, 0, 21.
- **Questions for your decision:** Is the full count known at initial booking? Which included changes happen later? Would this count leak future amendment or cancellation information when forecasting from the booking date?


In [13]:
show_columns = ['reserved_room_type', 'assigned_room_type', 'booking_changes']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,reserved_room_type,object,0,"C, A, D, E"
1,assigned_room_type,object,0,"C, A, D, E"
2,booking_changes,int64,0,"3, 4, 0, 1"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "reserved_room_type": {"bin": None, "reason": ""},
    "assigned_room_type": {"bin": None, "reason": ""},
    "booking_changes": {"bin": None, "reason": ""},
})


### Cell group 13 — Group 8 — Deposit and booking partner IDs

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Are payment and partner values guaranteed to be known and fixed at booking creation in this export? What would you need to verify?

### Paper evidence for these fields

### 23. `deposit_type` — DepositType (categorical)

- **Paper definition:** Whether a deposit was paid: No Deposit; Non Refund (deposit equals total stay cost); Refundable (deposit is less than total cost).
- **Source / engineering:** BO and TR (Transactions). Based on payments found before arrival or cancellation. If no payment is found, value is No Deposit; if payments equal/exceed total stay cost, Non Refund; otherwise Refundable.
- **Published profile:** H1: 3 unique values; No Deposit 38,199; Non-Refundable 1,719; Refundable 142. H2: 3; No Deposit 66,442; Non-Refundable 12,868; Refundable 20.
- **Questions for your decision:** Can payment occur after booking creation? Is the classification calculated using payments through a later date? Does the “before cancellation date” wording mean canceled bookings have a different information window? What would a timestamped payment snapshot show?

### 24. `agent` — Agent (categorical identifier)

- **Paper definition:** ID of the travel agency that made the booking. The ID is shown instead of the agency name for anonymity.
- **Source / engineering:** BO and BL.
- **Published profile:** H1 has 186 unique values; top: 240 13,095; NULL 8,209; 250 2,869; 241 1,721. H2 has 224; top: 9 31,955; NULL 8,131; 1 7,137; 14 3,640.
- **Paper missing-value note:** NULL means “not applicable” (such as no travel agent), not a missing-value error.
- **Questions for your decision:** Is the agent known when the booking enters the system? Can it change if the booking is transferred or amended? Does the ID encode useful information, or is it only an anonymized category?

### 25. `company` — Company (categorical identifier)

- **Paper definition:** ID of the company/entity that made the booking or is responsible for paying; the ID is anonymized.
- **Source / engineering:** BO and BL.
- **Published profile:** H1 has 236 unique values; top: NULL 36,952; 223 784; 281 138; 154 133. H2 has 208; top: NULL 75,641; 40 924; 67 267; 45 250.
- **Paper missing-value note:** NULL is a “not applicable” category, not a missing-value error.
- **Questions for your decision:** When is the payer/company link established? Could it change later? How should the large NULL category be interpreted without treating it as missing?


In [14]:
show_columns = ['deposit_type', 'agent', 'company']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,deposit_type,object,0,"No Deposit, Refundable, Non Refund"
1,agent,float64,10676,"304.0, 240.0, 303.0, 15.0"
2,company,float64,74216,"110.0, 113.0, 270.0, 178.0"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "deposit_type": {"bin": None, "reason": ""},
    "agent": {"bin": None, "reason": ""},
    "company": {"bin": None, "reason": ""},
})


### Cell group 14 — Group 9 — Wait, customer category, and rate

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Is each value immediately known at booking time, or derived/updated later? What timestamps would settle the question?

### Paper evidence for these fields

### 26. `days_in_waiting_list` — DaysInWaitingList (integer)

- **Paper definition:** Number of days a booking remained on the waiting list before being confirmed to the customer.
- **Source / engineering:** BO; calculated as confirmation date minus the date the booking entered the PMS.
- **Published profile:** H1: 0.53, 7.43, 0, 0, 0, 0, 185. H2: 3.23, 20.87, 0, 0, 0, 0, 391.
- **Questions for your decision:** Is this known when the request first enters the system, or only once confirmed? Is the “booking” timestamp before or after confirmation? What date would you use for a prediction made at confirmation?

### 27. `customer_type` — CustomerType (categorical)

- **Paper definition:** Contract (allotment or other contract); Group; Transient (not part of a group/contract or linked transient booking); Transient-party (transient booking associated with at least one other transient booking).
- **Source / engineering:** BO and BL.
- **Published profile:** H1 has 4 values; Transient 30,209; Transient-party 7,791; Contract 1,776; Group 284. H2: Transient 59,404; Transient-party 17,333; Contract 2,300; Group 293.
- **Questions for your decision:** When is the booking's group/contract relationship known? Could the association be added later? Is Transient-party derived from other bookings in a way that may not exist at the initial booking time?

### 28. `adr` — ADR (numeric)

- **Paper definition:** Average Daily Rate as defined by reference [5]. In this dataset it is lodging transaction amount divided by total nights stayed.
- **Source / engineering:** BO, BL, and TR; calculated from the sum of lodging transactions and total stay nights.
- **Published profile:** H1: 94.95, 61.44, -6.38, 50, 75, 125, 508. H2: 105.3, 43.6, 0, 79.2, 99.9, 126, 5,400.
- **Questions for your decision:** Is ADR fixed when first booked or can rate/transactions/stay length change? Why are negative and very large values present in the published profile? How will you investigate them without silently changing rows? The course contract uses ADR in the empty-room stay-value calculation, so distinguish that scoring use from predictor timing.


In [15]:
show_columns = ['days_in_waiting_list', 'customer_type', 'adr']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,days_in_waiting_list,int64,0,"0, 50, 47, 65"
1,customer_type,object,0,"Transient, Contract, Transient-Party, Group"
2,adr,float64,0,"0.0, 75.0, 98.0, 107.0"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "days_in_waiting_list": {"bin": None, "reason": ""},
    "customer_type": {"bin": None, "reason": ""},
    "adr": {"bin": None, "reason": ""},
})


### Cell group 15 — Group 10 — Requests and final status

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Can requests accumulate? Which fields directly expose a cancellation, no-show, checkout, or the date the outcome was set?

### Paper evidence for these fields

### 29. `required_car_parking_spaces` — RequiredCardParkingSpaces (integer)

- **Paper definition:** Number of car-parking spaces required by the customer.
- **Source / engineering:** BO and BL.
- **Published profile:** H1: 0.14, 0.35, 0, 0, 0, 0, 8. H2: 0.024, 0.15, 0, 0, 0, 0, 3.
- **Questions for your decision:** Is this requested at reservation time or can the request be added later? How should zero be interpreted? Does the final count match a booking-time snapshot?

### 30. `total_of_special_requests` — TotalOfSpecialRequests (integer)

- **Paper definition:** Total number of customer special requests, such as a twin bed or high floor.
- **Source / engineering:** BO and BL; sum of all special requests.
- **Published profile:** H1: 0.62, 0.81, 0, 0, 0, 1, 5. H2: 0.55, 0.78, 0, 0, 0, 1, 5.
- **Questions for your decision:** Can requests accumulate or change after the initial reservation? Does the source preserve when each request was made? At what date would the displayed total be available?

### 31. `reservation_status` — ReservationStatus (categorical outcome)

- **Paper definition:** Last reservation status, with three categories: Canceled (customer canceled); Check-Out (customer checked in and later departed); No-Show (customer did not check in and informed the hotel of the reason).
- **Source / engineering:** BO.
- **Published profile:** H1 has 3 values; Check-Out 28,938; Canceled 10,831; No-Show 291. H2: Check-Out 46,228; Canceled 32,186; No-Show 916.
- **Questions for your decision:** Is this a final outcome written after the decision? How does `No-Show` relate to the course contract's `is_canceled` target? Should this be a predictor, target-adjacent check, or validation field only?

### 32. `reservation_status_date` — ReservationStatusDate (date)

- **Paper definition:** Date the last status was set; used with status to understand when a booking was canceled or when the guest checked out.
- **Source / engineering:** BO.
- **Published profile:** H1: min 2014-11-18; max 2017-09-14; median 2016-07-31; 913 unique dates. H2: min 2014-10-17; max 2017-09-07; median 2016-08-10; 864 unique dates.
- **Questions for your decision:** Does this date record the outcome or its final update? Would it be known at original booking time? Could it disclose cancellation/check-out timing and therefore leak the target?


In [16]:
show_columns = ['required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
evidence_rows = []

for column in show_columns:
    values = train_choose[column]
    examples = []

    # Collect up to four example values without using proof-period rows.
    for value in values.dropna():
        text_value = str(value)
        if text_value not in examples:
            examples.append(text_value)
        if len(examples) == 4:
            break

    evidence_rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": values.isna().sum(),
        "example_values_train_choose": ", ".join(examples),
    })

evidence_table = pd.DataFrame(evidence_rows)
display(evidence_table)


,column,data_type,missing_train_choose,example_values_train_choose
0,required_car_parking_spaces,int64,0,"0, 1, 2, 3"
1,total_of_special_requests,int64,0,"0, 1, 3, 2"
2,reservation_status,object,0,"Check-Out, Canceled, No-Show"
3,reservation_status_date,object,0,"2015-07-01, 2015-07-02, 2015-07-03, 2015-05-06"


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "required_car_parking_spaces": {"bin": None, "reason": ""},
    "total_of_special_requests": {"bin": None, "reason": ""},
    "reservation_status": {"bin": None, "reason": ""},
    "reservation_status_date": {"bin": None, "reason": ""},
})


## Final cell — Check your work and save your decisions

Run this only after you have completed every decision cell. It checks that all 32 fields have a valid bin and a reason, then writes a separate review file. It will not fill gaps or choose bins for you.


In [ ]:
expected_columns = list(bookings.columns)
missing_answers = []
extra_answers = []
invalid_answers = []
valid_bins = [1, 2, 3]

for column in expected_columns:
    if column not in my_decisions:
        missing_answers.append(column)

for column in my_decisions:
    if column not in expected_columns:
        extra_answers.append(column)

for column in my_decisions:
    answer = my_decisions[column]
    bin_number = answer["bin"]
    reason = answer["reason"]
    if bin_number not in valid_bins or reason.strip() == "":
        invalid_answers.append(column)

if missing_answers or extra_answers or invalid_answers:
    print("Finish or correct your decisions before saving.")
    print("Missing fields:", missing_answers)
    print("Unexpected fields:", extra_answers)
    print("Missing bin or reason:", invalid_answers)
else:
    bin_names = {
        1: "Fixed at booking",
        2: "Amendable or timing uncertain",
        3: "After the fact",
    }
    report_rows = []

    for column in expected_columns:
        bin_number = my_decisions[column]["bin"]
        reason = my_decisions[column]["reason"]
        report_rows.append({
            "column": column,
            "your_bin": bin_number,
            "bin_name": bin_names[bin_number],
            "your_reason": reason,
        })

    report = pd.DataFrame(report_rows)
    display(report)

    output_path = project_root / "docs" / "COLUMNS_USER_REVIEW.md"
    lines = [
        "# User-Completed Column Timing Audit",
        "",
        "> Decisions entered by the learner in `notebooks/01_column_audit_workshop.ipynb`.",
        "",
        "| Column | Bin | Reason in my words |",
        "|---|---:|---|",
    ]

    for column in expected_columns:
        bin_number = my_decisions[column]["bin"]
        reason = my_decisions[column]["reason"]
        reason = reason.replace("|", "\\|")
        reason = reason.replace("\n", " ")
        line = f"| `{column}` | {bin_number} - {bin_names[bin_number]} | {reason} |"
        lines.append(line)

    output_text = "\n".join(lines) + "\n"
    output_path.write_text(output_text, encoding="utf-8")
    print("Saved your review to:", output_path)
